# Hopefully we'll learn some general patterns with the herding data, and special cases with forgetting events

In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm 

from loka.data import load_breakthrough_states, save_breakthrough_states, load_forgetting_states, save_forgetting_states
from loka.models import BreakthroughCNN
from loka.training_methods import ForgettingScores

import os
import pandas as pd
from dotenv import load_dotenv

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

load_dotenv()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

BASE = os.getenv("BASE_DIR")
X_h,Y_h = load_breakthrough_states(f"{BASE}/data/ordered-data/herding_25p.txt")
X_c,Y_c,C_c = load_forgetting_states(f"{BASE}/data/forgetting_counts/forgetting_e.txt")

# X_eval, Y_eval = load_breakthrough_states(f"{BASE}/data/game_states/o.txt")

order = (torch.topk(torch.tensor(C_c),len(C_c))).indices
order = torch.Tensor.numpy(order)
order

In [ ]:
BATCH_SIZE = 512
TOTAL_EPOCHS = 50

X_c = torch.FloatTensor(X_c).to(device)
Y_c = torch.FloatTensor(Y_c).to(device)

X_h = torch.FloatTensor(X_h).to(device)
Y_h = torch.FloatTensor(Y_h).to(device)

#X_eval = torch.FloatTensor(X_eval).to(device)
#Y_eval = torch.FloatTensor(Y_eval).to(device)


def train(model, optimizer, loss_fn, X, Y, epochs, pbar, pbar_inc, n_points):
    X_t = X[:n_points].to(device)
    Y_t = Y[:n_points].to(device)

    for _ in range(epochs):
        perm = torch.randperm(len(X_t), device=device)
        
        for i in range(0, len(X_t), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb, yb = X_t[idx], Y_t[idx]

            optimizer.zero_grad()
            batch_loss = loss_fn(model(xb), yb)
            batch_loss.backward()
            optimizer.step()
        
        pbar.update(pbar_inc)

all_mse_results = list()
all_prc_results = list()

ratios = [0.8, 0.6, 0.4, 0.2]
start, end, steps = 0.01, 0.1, 50
splits = np.linspace(start, end, num=steps, endpoint=True)

# work per split, computed up front so the bar total is exact
n_points_per_split = [int(s * len(X_c)) for s in splits]
total_work = len(ratios) * TOTAL_EPOCHS * sum(n_points_per_split)

pbar = tqdm(total=total_work, desc='Training', bar_format="{l_bar}{bar} {percentage:3.0f}% [{elapsed}<{remaining}]")

for ratio in ratios:
    herding_epochs = int(ratio * TOTAL_EPOCHS)
    forgetting_epochs = int(round((1-ratio) * TOTAL_EPOCHS,2))

    mse_values = list()
    prc_values = list()

    for s, pbar_inc in zip(splits, n_points_per_split):

        n_points = int(s * len(X_c)) # X_c is everything

        torch.manual_seed(SEED)
        model = BreakthroughCNN().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.MSELoss()

        # train first with herding data
        train(model, optimizer, loss_fn, X_h, Y_h, herding_epochs, pbar, pbar_inc, n_points)

        # train then with the forgetting count
        train(model, optimizer, loss_fn, X_c[order], Y_c[order], forgetting_epochs, pbar, pbar_inc, n_points)
        
        with torch.no_grad():
            test_preds = model(X_c)
            mse = loss_fn(test_preds, Y_c).item()
            mse_values.append(mse)
            prc_values.append(s)
        
        del model, optimizer
        torch.cuda.empty_cache()

        # dev break
        if s > 0.02:
            break

    all_mse_results.append(mse_values)
    all_prc_results.append(prc_values)
    
    # dev break
    break

    

In [ ]:
data_dir = f"{BASE}/data/forgetting_counts/herding_and_forgetting"
os.makedirs(data_dir, exist_ok=True)  # create the folder if it doesn't exist yet

csv_file = f'{data_dir}/k_center_no_S0_25p_correct_order.csv'

for i, ratio in enumerate(ratios):
    prc_values = all_prc_results[i]
    mse_values = all_mse_results[i]

    h = int(round(ratio * 100))
    csv_file = f"herding_{h}_forgetting_{100 - h}.csv" 
    
    pd.DataFrame({'ratio': prc_values, 'mse': mse_values}).to_csv(csv_file, index=False)